# Kütüphaneler & Veri

In [ ]:
# Gerekli paketlerin kurulumu
!pip install -q datasets
!pip install --upgrade -q datasets
!pip install -q openai

# Gerekli kütüphanelerin import edilmesi
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from datasets import load_dataset
import warnings
import random
import pandas as pd
import time
import matplotlib.pyplot as plt

# Uyarıları görmezden gel
warnings.filterwarnings('ignore')

In [ ]:
dataset = pd.read_csv("gsm8k_tr.csv")

random.seed(404142)

selected_pairs = random.sample(list(zip(dataset['question'], dataset['answer'])), 1500)
questions = [question for question, _ in selected_pairs]
answers = [answer for _, answer in selected_pairs]

# OpenAI API

In [ ]:
client = OpenAI(api_key="xxx_APIKEY_xxx")

client

# Attribute Üretme

In [ ]:
def generate_question_attributes(question, client):
    """
    OpenAI API kullanarak verilen soru için 3 adet metin özniteliği üretir.

    Args:
        question: İşlenecek soru metni.
        client: OpenAI API client nesnesi.

    Returns:
        Üretilen 3 adet metin özniteliği.
    """
    try:
        response = client.chat.completions.create(
            #model="gpt-3.5-turbo",
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": """
                Sen bir metin analiz uzmanısın.
                Sana verilen matematik soruları için, sorunun içeriğinden tamamen bağımsız olarak 3 kısa açıklama üret.
                Bu açıklamalar şunları içermelidir:
                - Sorunun matematiksel türü ve yapısı.
                - Sorunun dilsel özellikleri (örneğin, sayı, bilinmeyen, oran vb. içerip içermediği).
                - Sorunun çözüm yöntemiyle ilgili genel bilgiler.

                Ürettiğin açıklamalar, sorunun içeriğine özgü bilgi (örneğin, kişi isimleri, belirli nesneler veya tam sayılar) içermemelidir.
                Benzer sorular için benzer açıklamalar üretmelisin ve tüm yanıtlar aynı formatta olmalıdır.

                İyi ve kötü örnekler aşağıda verilmiştir:

                **İyi Örnek:**
                Soru: Bir kişi 2500 dolara bir squat rack satın alır. Halterin fiyatı ise 1/10 kadardı. Her şey için ne kadar ödedi?
                Özellikler:
                1) Bu soru oran-orantı türünde bir problemdir. İki miktar arasındaki oranı kullanarak bir değer hesaplanması gerekir.
                2) Soruda bir miktarın başka bir miktarla ilişkisi ifade edilmiştir. Hesaplama için bu ilişki kullanılmalıdır.
                3) Çözüm, oranı belirlemek, verilen miktarı kullanarak diğer miktarı hesaplamak ve sonuçları toplamak şeklindedir.

                **Kötü Örnek:**
                Soru: Bir kişi 2500 dolara bir squat rack satın alır. Halterin fiyatı ise 1/10 kadardı. Her şey için ne kadar ödedi?
                Özellikler:
                1) Soruda squat rack ve halter fiyatları verilmiştir. Bu iki fiyatın toplamı bulunmalıdır.
                2) Squat rack'in fiyatı halterin fiyatından 10 kat fazladır.
                3) Çözüm, squat rack ve halter fiyatlarını toplayarak bulunur.

                (Kötü örnekte squat rack ve halter gibi soru içeriğine özgü bilgiler yer almaktadır. Bu istenmeyen bir durumdur.)

                **Format:**
                1) <Açıklama>
                2) <Açıklama>
                3) <Açıklama>
                """},
                {"role": "user", "content": f"""
                Soru: {question}
                Bu soru için 3 kısa açıklama üretin. Her açıklama yeni bir satırda olmalıdır.

                Format:
                1) <Açıklama>
                2) <Açıklama>
                3) <Açıklama>
                """},
            ]
        )
        attributes = response.choices[0].message.content.strip().split('\n')
        # Boşlukları ve gereksiz karakterleri temizleme
        attributes = [attr.strip().lstrip('-').strip() for attr in attributes if attr.strip()]
        # Numaralandırmaları kaldırma (örneğin, "1)", "2)", "3)")
        attributes = [
            attr[attr.find(")")+1:].strip() if ")" in attr else attr for attr in attributes
        ]
        # 3 özellik kontrolü
        while len(attributes) < 3:
            attributes.append("Ek özellik belirlenemedi.")
        return attributes[:3]
    except Exception as e:
        print(f"Error generating attributes for question: {question}\nError: {e}")
        return ["Özellik üretilemedi", "Özellik üretilemedi", "Özellik üretilemedi"]

In [ ]:
chunk_size = 500  # Her chunk'ta işlenecek satır sayısı
total_questions = len(dataset['train']['question'])  # Toplam soru sayısı

# Sonuçları saklamak için boş bir liste
all_attributes = []

# Chunk'lar halinde soruları işleme
for i in range(0, total_questions, chunk_size):

    questions_chunk = dataset['train']['question'][i:i + chunk_size]

    # Bu chunk'taki sorular için öznitelikleri oluştur
    for question in questions_chunk:
        attributes = generate_question_attributes(question, client)
        all_attributes.append({
            "question": question,
            "attribute_1": attributes[0],
            "attribute_2": attributes[1],
            "attribute_3": attributes[2]
        })

    print(f"{i + chunk_size} soruya kadar işlendi...")  # İşlem durumunu takip

attributes_df = pd.DataFrame(all_attributes)

attributes_df.to_csv("question_attributes.csv", index=False)

print("Tüm sorular işlendi ve sonuçlar kaydedildi.")

# Embedding

In [ ]:
# Yeni bir sütun oluşturma ve formatlama
qa['combined_qa'] = (
    "Soru:\n" + qa['question'] + "\n\n" +  # Soru kısmı
    "Üretilmiş Metinler:\n" +
    "1) " + qa['attribute_1'] + "\n" +    # Attribute 1
    "2) " + qa['attribute_2'] + "\n" +    # Attribute 2
    "3) " + qa['attribute_3'] + "\n"      # Attribute 3
)

combined_qa_list = qa['combined_qa'].tolist()

In [ ]:
def get_embeddings_in_batches(texts, tokenizer, model, batch_size=16):
    embeddings = []
    model = model.to("cuda")  # Modeli GPU'ya taşı
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to("cuda")  # Girişleri GPU'ya taşı

        with torch.no_grad():
            outputs = model(**inputs)  # Model GPU üzerinde çalıştırılıyor

        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # Sonuçları CPU'ya taşı
        embeddings.append(batch_embeddings)
        print(f"Batch {i} işleniyor")
    return np.vstack(embeddings)  # Batch'leri birleştir

In [ ]:
combined_qa_embeddings = get_embeddings_in_batches(combined_qa_list, tokenizer, model, batch_size=128)

# En Benzeyen Sorular

In [ ]:
num_similar_questions = 5
most_similar_questions = {}

# Her bir embedding için diğer tüm embeddinglerle kosinüs benzerliği
similarity_matrix = cosine_similarity(combined_qa_embeddings)

# Her soru için (kendi hariç) en yüksek benzerliğe sahip ilk N soru
for i in range(len(combined_qa_list)):
    # Benzerlikler, kendisi hariç (-1), indekslerle birlikte
    similarities = list(enumerate(similarity_matrix[i]))

    # Benzerliklere göre azalan sıra
    # enumerate(similarity_matrix[i]): [(0, 0.999), (1, 0.75), (2, 0.88), ...]
    # lambda item: item[1]: Sıralama anahtarı olarak benzerlik değerini (item[1])
    # reverse=True: Azalan sırada
    # [1:]: Kendisiyle olan benzerliği (indeks i, değer 1.0) atla
    sorted_similarities = sorted(similarities, key=lambda item: item[1], reverse=True)[1:]

    # En benzer N sorunun indeksleri ve benzerlik skorları
    top_n_indices_scores = sorted_similarities[:num_similar_questions]

    # İlgili soru metinlerini saklama
    question_text = qa.iloc[i]['question']
    similar_questions_list = []
    for index, score in top_n_indices_scores:
        similar_question_text = qa.iloc[index]['question']
        similar_questions_list.append({
            "question": similar_question_text,
            "similarity_score": score
        })

    most_similar_questions[question_text] = similar_questions_list

# Sonuçları görüntüleme (İlk birkaç soru için)
print("Her soru için en benzer 5 soru:")
count = 0
for original_question, similar_list in most_similar_questions.items():
    if count >= 5:  # Sadece ilk 5 soruyu göster
        break
    print(f"\nOriginal Question: {original_question}")
    print("Most Similar Questions:")
    for item in similar_list:
        print(f"- {item['question']} (Benzerlik Skoru: {item['similarity_score']:.4f})")
    count += 1

# DataFrame formatı: Original Question, Similar Question 1, Score 1, Similar Question 2, Score 2, ...
similar_questions_df_data = []
for original_question, similar_list in most_similar_questions.items():
    row = {"Original Question": original_question}
    for i, item in enumerate(similar_list):
        row[f"Similar Question {i+1}"] = item['question']
        row[f"Score {i+1}"] = item['similarity_score']
    similar_questions_df_data.append(row)

most_similar_questions_df = pd.DataFrame(similar_questions_df_data)

# DataFrame'i CSV olarak kaydedebilirsiniz (isteğe bağlı)
most_similar_questions_df.to_csv("most_similar_questions.csv", index=False)

# Turkish-Llama-8b-DPO-v0.1 Modeli

In [ ]:
# Model ve tokenizer yükleme
model_name = "ytu-ce-cosmos/Turkish-Llama-8b-DPO-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Dynamic 5-Shot

In [ ]:
comparison_results_incontext = []

# Cihazı otomatik olarak belirleme (GPU varsa kullanılır, yoksa CPU kullanılır)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modeli cihaza taşıma
model.to(device)

start_time = time.time()

for i in range(1500):
    if i in [0, 9, 99, 499, 799, 999, 1299, 1399]:
        print(f"Iterasyon {i+1} zamanı: {time.time() - start_time:.2f} saniye")

    question = questions[i]
    true_answer = answers[i]

    # Soru için en benzer 5 soruyu kullanarak prompt oluşturma
    similar_row = most_similar_questions_df[most_similar_questions_df['Original Question'] == question].iloc[0]
    incontext_examples = ""
    for j in range(1, 4-6):  # Similar Question 1'den Similar Question 5'e kadar
        similar_question = similar_row[f'Similar Question {j}']
        similar_answer_row = dataset[dataset['question'] == similar_question].iloc[0]
        similar_answer = similar_answer_row['answer']
        incontext_examples += f"Soru: {similar_question}\nCevap: {similar_answer}\n\n"

    # Ana soru için prompt'u oluşturma
    prompt = f"Aşağıdaki matematik sorusunu çöz ve cevabını getir:\n\n"
    prompt += incontext_examples
    prompt += f"Soru: {question}\nCevap:"

    try:
        # Prompt'u token'ize etme ve modele gönderme
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(device)
        output_tokens = model.generate(
            input_ids,
            max_new_tokens=100,  # Üretilecek maksimum token sayısı
            num_return_sequences=1,
            do_sample=True,
            top_k=10,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id  # Padding için EOS token kullanılması
        )
        predicted_answer = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    except Exception as e:
        predicted_answer = f"Tahmin hatası: {e}"
        print(f"Hata oluştu: {e}")

    comparison_results_incontext.append({
        "Question": question,
        "True Answer": true_answer,
        "Predicted Answer (In-context)": predicted_answer,
    })

end_time = time.time()
print(f"Kod tamamlandı! Toplam süre: {end_time - start_time:.2f} saniye")

# Random 5-Shot

In [ ]:
comparison_results_random_incontext = []

# Cihazı otomatik olarak belirleme (GPU varsa kullanılır, yoksa CPU kullanılır)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modeli cihaza taşıma
model.to(device)

start_time = time.time()

for i in range(1500):
    if i in [0, 9, 99, 499, 799, 999, 1299, 1399]:
        print(f"Iterasyon {i+1} zamanı: {time.time() - start_time:.2f} saniye")

    question = questions[i] # Daha önce seçilen rastgele 1500 sorudan biri
    true_answer = answers[i] # Karşılık gelen doğru cevap

    # In-context learning için rastgele 5 soru-cevap çifti seçme
    # Tamamen rastgele seçim yapıyoruz, benzerliğe bakmıyoruz.
    if isinstance(dataset, pd.DataFrame):
      random_indices = random.sample(range(len(dataset)), 5)
      random_examples = dataset.iloc[random_indices]
    elif isinstance(dataset, dict) and 'train' in dataset: # Hugging Face Dataset
      random_indices = random.sample(range(len(dataset['train'])), 5)
      random_examples = dataset['train'].select(random_indices)
    else:
      raise TypeError("Desteklenmeyen dataset formatı. Lütfen pandas DataFrame veya Hugging Face Dataset sağlayın.")

    incontext_examples = ""
    for index, row in random_examples.iterrows() if isinstance(dataset, pd.DataFrame) else enumerate(random_examples):
        # DataFrame için row['question'], Hugging Face Dataset için row['question']
        ex_question = row['question']
        ex_answer = row['answer']
        incontext_examples += f"Soru: {ex_question}\nCevap: {ex_answer}\n\n"

    # Ana soru için prompt'u oluşturma
    prompt = f"Aşağıdaki matematik sorusunu çöz ve cevabını getir:\n\n"
    prompt += incontext_examples # Rastgele 5 soru-cevap örneği
    prompt += f"Soru: {question}\nCevap:" # Çözülmesi istenen asıl soru

    try:
        # Prompt'u token'ize etme ve modele gönderme
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(device)

        # Modelden tahmin alma
        output_tokens = model.generate(
            input_ids,
            max_new_tokens=100,  # Üretilecek maksimum token sayısı
            num_return_sequences=1,
            do_sample=True,
            top_k=10,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id  # Padding için EOS token kullanılması
        )
        predicted_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

        # Decode edilen metinden sadece modelin ürettiği cevabı çıkarmaya çalışma
        # Prompt metnini decode edilen metinden çıkar
        predicted_answer = predicted_text[len(prompt):].strip()


    except Exception as e:
        predicted_answer = f"Tahmin hatası: {e}"
        print(f"Hata oluştu: {e}")

    comparison_results_random_incontext.append({
        "Question": question,
        "True Answer": true_answer,
        "Predicted Answer (Random In-context)": predicted_answer,
    })

end_time = time.time()
print(f"Kod tamamlandı! Toplam süre: {end_time - start_time:.2f} saniye")


# Listeyi DataFrame'e çevirme
comparison_results_random_df = pd.DataFrame(comparison_results_random_incontext)

# DataFrame'i CSV dosyasına kaydetme
comparison_results_random_df.to_csv("comparison_results_random_incontext.csv", index=False)

print("comparison_results_random_incontext.csv dosyası başarıyla oluşturuldu ve kaydedildi.")

# Zero-Shot

In [ ]:
comparison_results_zeroshot = []

# Cihazı otomatik olarak belirleme (GPU varsa kullanılır, yoksa CPU kullanılır)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modeli cihaza taşıma
model.to(device)

start_time = time.time()

for i in range(1500):
    if i in [0, 9, 99, 499, 799, 999, 1299, 1399]:
        print(f"Iterasyon {i+1} zamanı: {time.time() - start_time:.2f} saniye")

    question = questions[i] # Daha önce seçilen rastgele 1500 sorudan biri
    true_answer = answers[i] # Karşılık gelen doğru cevap

    # Zero-shot prompt oluşturma
    # Sadece soruyu verip cevabı bulmasını istiyoruz.
    prompt = f"Aşağıdaki matematik sorusunu çöz ve cevabını getir:\n\nSoru: {question}\nCevap:"

    try:
        # Prompt'u token'ize etme ve modele gönderme
        # max_length sadece prompt uzunluğunu kapsayacak kadar olmalı.
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(device)

        # Modelden tahmin alma
        output_tokens = model.generate(
            input_ids,
            max_new_tokens=100,  # Üretilecek maksimum token sayısı
            num_return_sequences=1,
            do_sample=True,
            top_k=10,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id  # Padding için EOS token kullanılması
        )
        predicted_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

        # Decode edilen metinden sadece modelin ürettiği cevabı çıkarmaya çalışma
        # Prompt metnini decode edilen metinden çıkar
        predicted_answer = predicted_text[len(prompt):].strip()

    except Exception as e:
        predicted_answer = f"Tahmin hatası: {e}"
        print(f"Hata oluştu: {e}")

    comparison_results_zeroshot.append({
        "Question": question,
        "True Answer": true_answer,
        "Predicted Answer (Zero-Shot)": predicted_answer,
    })

end_time = time.time()
print(f"Kod tamamlandı! Toplam süre: {end_time - start_time:.2f} saniye")

# Listeyi DataFrame'e çevirme
comparison_results_zeroshot_df = pd.DataFrame(comparison_results_zeroshot)

# DataFrame'i CSV dosyasına kaydetme
comparison_results_zeroshot_df.to_csv("comparison_results_zeroshot.csv", index=False)

print("comparison_results_zeroshot.csv dosyası başarıyla oluşturuldu ve kaydedildi.")

comparison_results_zeroshot

# Doğrudan Cevaba Ulaşmak

In [ ]:
def get_direct_answer(answer_text, client):
    """
    OpenAI API kullanarak matematiksel çözüm metninden direkt sayısal cevabı çeker.

    Args:
        answer_text: Matematiksel çözüm metni.
        client: OpenAI API client nesnesi.

    Returns:
        Çözüm metninden çıkarılan direkt sayısal cevap veya hata mesajı.
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": """
                Sana bir matematik sorusunun adım adım çözümü ve nihai cevabını içeren bir metin verilecektir.
                Senin görevin, bu metinden sadece nihai sayısal cevabı (tam sayı veya ondalık) çıkarmak ve bu cevabı direkt olarak döndürmektir.
                Çözüm adımlarını veya metinsel açıklamaları dahil etme. Cevabı sadece sayısal değeri döndür.
                Eğer metinde belirgin bir sayısal nihai cevap yoksa, "Çözüm bulunamadı" gibi bir mesaj döndür.
                Cevap "on elma" gibi metin bazlı olabilir, böyle durumda da 10 cevabını döndür.

                Örnek:
                Soru: Mira her sabah koşar. Saatte 5 mil koşuyor. Her sabah 2 saat koşarsa, beş gün boyunca kaç mil koşabilir?
                Gerçek Cevap: Mira günde 5 mil/saat * 2 saat = 10 mil koşar. Beş gün boyunca 10 mil/gün * 5 gün = 50 mil koşar.
                Direkt Cevap: 50

                Örnek 2:
                Soru: Bir markette elmalar 2 TL, muzlar 3 TL'dir. 3 elma ve 2 muz alan bir kişi kaç TL öder?
                Gerçek Cevap: 3 elma için 3 * 2 = 6 TL. 2 muz için 2 * 3 = 6 TL. Toplam ödeme 6 + 6 = 12 TL.
                Direkt Cevap: 12

                Lütfen sadece nihai sayısal cevabı verin.
                """},
                {"role": "user", "content": f"""
                Aşağıdaki matematiksel çözüm metninden sadece direkt sayısal cevabı çıkarın:
                {answer_text}

                Direkt Cevap:
                """},
            ],
            max_tokens=50 # Sadece sayısal cevap beklendiği için düşük token sayısı yeterli
        )
        direct_answer = response.choices[0].message.content.strip()
        # Yanıtın sayısal olup olmadığını kontrol etme (basit bir kontrol)
        try:
            float(direct_answer)
            return direct_answer
        except ValueError:
            # Sayısal değilse veya beklenmedik bir formatta ise
            return "Çözümden sayısal cevap çıkarılamadı"

    except Exception as e:
        print(f"Error extracting direct answer: {e}")
        return f"API Hatası: {e}"

In [ ]:
all_direct_answers = []
start_time = time.time()

for i, ans in enumerate(answers):
  # İlk 1, 5, 10, 50, 100 ve sonuncu (1500) iterasyonların zamanını yazdır
  if i in [0, 4, 9, 49, 99, 1499]:
    print(f"Direkt Cevap Çıkarma - Iterasyon {i+1} zamanı: {time.time() - start_time:.2f} saniye")

  direct_ans = get_direct_answer(ans, client)
  all_direct_answers.append(direct_ans)

In [ ]:
dynamic_5s = pd.read_csv("comparison_results_incontext.csv")
random_5s = pd.read_csv("comparison_results_random_incontext.csv")
zeroshot = pd.read_csv("comparison_results_zeroshot.csv")

# dynamic_5s dataframe'indeki 'Predicted Answer (In-context)' sütunundaki her bir cevabın direkt sayısal kısmını çıkar
dynamic_5s['Predicted Direct Answer (In-context)'] = dynamic_5s['Predicted Answer (In-context)'].apply(lambda x: get_direct_answer(x, client))

# random_5s dataframe'indeki 'Predicted Answer (Random In-context)' sütunundaki her bir cevabın direkt sayısal kısmını çıkar
random_5s['Predicted Direct Answer (Random In-context)'] = random_5s['Predicted Answer (Random In-context)'].apply(lambda x: get_direct_answer(x, client))

# zeroshot dataframe'indeki 'Predicted Answer (Zero-Shot)' sütunundaki her bir cevabın direkt sayısal kısmını çıkar
zeroshot['Predicted Direct Answer (Zero-Shot)'] = zeroshot['Predicted Answer (Zero-Shot)'].apply(lambda x: get_direct_answer(x, client))

In [ ]:
# Gerçek cevaplardan sayısal cevap çıkarılamayanların sayısı
original_answers_no_numerical = all_direct_answers.count("Çözümden sayısal cevap çıkarılamadı") + all_direct_answers.count("API Hatası: OpenAI API returned None") # API Hatası mesajlarını da dahil et

# Cevap Analizi

In [ ]:
# Ana DataFrame
final_comparison_df = pd.DataFrame({
    'Gerçek Soru': questions,  # Daha önce random.sample ile seçilen 1500 soru
    'Gerçek Cevap': answers,  # Karşılık gelen 1500 gerçek cevap
    'Gerçek Cevap Direkt Sayısal': all_direct_answers  # get_direct_answer ile çıkarılan 1500 direkt cevap
})

# Dynamic 5-Shot sonuçlarını birleştirme
final_comparison_df = pd.merge(
    final_comparison_df,
    dynamic_5s[['Question', 'Predicted Answer (In-context)', 'Predicted Direct Answer (In-context)']],
    left_on='Gerçek Soru',
    right_on='Question',
    how='left'
)
final_comparison_df = final_comparison_df.drop('Question', axis=1) # Birleştirme sütununu kaldır

# Random 5-Shot sonuçlarını birleştirme
final_comparison_df = pd.merge(
    final_comparison_df,
    random_5s[['Question', 'Predicted Answer (Random In-context)', 'Predicted Direct Answer (Random In-context)']],
    left_on='Gerçek Soru',
    right_on='Question',
    how='left'
)
final_comparison_df = final_comparison_df.drop('Question', axis=1) # Birleştirme sütununu kaldır

# Zero-Shot sonuçlarını birleştirme
final_comparison_df = pd.merge(
    final_comparison_df,
    zeroshot[['Question', 'Predicted Answer (Zero-Shot)', 'Predicted Direct Answer (Zero-Shot)']],
    left_on='Gerçek Soru',
    right_on='Question',
    how='left'
)
final_comparison_df = final_comparison_df.drop('Question', axis=1) # Birleştirme sütununu kaldır

final_comparison_df.to_csv("final_comparison_results.csv", index=False)
print("\nfinal_comparison_results.csv dosyası başarıyla oluşturuldu ve kaydedildi.")

In [ ]:
# Sayısal cevap çıkarılamayan satırları filtreleme
# Gerçek cevabın direkt sayısal değeri 'Çözümden sayısal cevap çıkarılamadı' veya 'API Hatası...' olmayan satırlar
valid_comparison_df = final_comparison_df[
    (final_comparison_df['Gerçek Cevap Direkt Sayısal'] != "Çözümden sayısal cevap çıkarılamadı") &
    (~final_comparison_df['Gerçek Cevap Direkt Sayısal'].str.startswith('API Hatası:'))
].copy() #

print(f"Toplam Karşılaştırılan Soru Sayısı (Sayısal Cevabı Olanlar): {len(valid_comparison_df)}")

def is_correct(true_direct_answer_str, predicted_direct_answer_str):
    try:
        # Her iki değeri de float'a çevirip karşılaştır
        return float(true_direct_answer_str) == float(predicted_direct_answer_str)
    except (ValueError, TypeError):
        # Eğer herhangi biri sayıya çevrilemiyorsa, yanlış kabul et
        return False

# Doğruluk kontrolü yapma
valid_comparison_df['Dynamic 5-Shot Doğru'] = valid_comparison_df.apply(
    lambda row: is_correct(row['Gerçek Cevap Direkt Sayısal'], row['Predicted Direct Answer (In-context)']),
    axis=1
)

valid_comparison_df['Random 5-Shot Doğru'] = valid_comparison_df.apply(
    lambda row: is_correct(row['Gerçek Cevap Direkt Sayısal'], row['Predicted Direct Answer (Random In-context)']),
    axis=1
)

valid_comparison_df['Zero-Shot Doğru'] = valid_comparison_df.apply(
    lambda row: is_correct(row['Gerçek Cevap Direkt Sayısal'], row['Predicted Direct Answer (Zero-Shot)']),
    axis=1
)

# Doğru ve Yanlış sayılarını hesaplama
total_valid_questions = len(valid_comparison_df)

dynamic_5s_correct = valid_comparison_df['Dynamic 5-Shot Doğru'].sum()
dynamic_5s_wrong = total_valid_questions - dynamic_5s_correct

random_5s_correct = valid_comparison_df['Random 5-Shot Doğru'].sum()
random_5s_wrong = total_valid_questions - random_5s_correct

zeroshot_correct = valid_comparison_df['Zero-Shot Doğru'].sum()
zeroshot_wrong = total_valid_questions - zeroshot_correct

# Yüzdelik hesaplama
dynamic_5s_correct_percentage = (dynamic_5s_correct / total_valid_questions) * 100 if total_valid_questions > 0 else 0
dynamic_5s_wrong_percentage = (dynamic_5s_wrong / total_valid_questions) * 100 if total_valid_questions > 0 else 0

random_5s_correct_percentage = (random_5s_correct / total_valid_questions) * 100 if total_valid_questions > 0 else 0
random_5s_wrong_percentage = (random_5s_wrong / total_valid_questions) * 100 if total_valid_questions > 0 else 0

zeroshot_correct_percentage = (zeroshot_correct / total_valid_questions) * 100 if total_valid_questions > 0 else 0
zeroshot_wrong_percentage = (zeroshot_wrong / total_valid_questions) * 100 if total_valid_questions > 0 else 0


# Sonuçları yazdırma
print("\n--- Doğruluk Karşılaştırması (Sayısal Cevap Çıkarılabilenler) ---")
print(f"Toplam Değerlendirilen Soru Sayısı: {total_valid_questions}")
print("-" * 40)

print("Dynamic 5-Shot Sonuçları:")
print(f"  Doğru Sayısı: {dynamic_5s_correct}")
print(f"  Yanlış Sayısı: {dynamic_5s_wrong}")
print(f"  Doğruluk Oranı: {dynamic_5s_correct_percentage:.2f}%")
print(f"  Yanlışlık Oranı: {dynamic_5s_wrong_percentage:.2f}%")
print("-" * 40)

print("Random 5-Shot Sonuçları:")
print(f"  Doğru Sayısı: {random_5s_correct}")
print(f"  Yanlış Sayısı: {random_5s_wrong}")
print(f"  Doğruluk Oranı: {random_5s_correct_percentage:.2f}%")
print(f"  Yanlışlık Oranı: {random_5s_wrong_percentage:.2f}%")
print("-" * 40)

print("Zero-Shot Sonuçları:")
print(f"  Doğru Sayısı: {zeroshot_correct}")
print(f"  Yanlış Sayısı: {zeroshot_wrong}")
print(f"  Doğruluk Oranı: {zeroshot_correct_percentage:.2f}%")
print(f"  Yanlışlık Oranı: {zeroshot_wrong_percentage:.2f}%")
print("-" * 40)

In [ ]:
# Doğruluk oranları (daha önceki kod bloklarından hesaplananlar)
labels = ['Dynamic 5-Shot', 'Random 5-Shot', 'Zero-Shot']
correct_percentages = [dynamic_5s_correct_percentage, random_5s_correct_percentage, zeroshot_correct_percentage]

x = np.arange(len(labels))  # Etiket konumları
width = 0.35  # Çubukların genişliği

fig, ax = plt.subplots(figsize=(8, 6))

# Doğruluk çubukları
rects1 = ax.bar(x, correct_percentages, width, label='Doğruluk', color='skyblue')

# Eksen etiketleri ve başlık
ax.set_ylabel('Doğruluk Oranı (%)')
ax.set_title('Doğruluk Oranları')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Çubukların üzerine değerleri yazdır
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate('%.2f%%' % height,
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)

fig.tight_layout()
plt.show()